# LUDB — download, convert, label, visualize

[Lobachevsky University ECG Database](https://physionet.org/content/ludb/1.0.1/) (200 × 10 s, 12-lead, 500 Hz).

Already-processed records are skipped (no PhysioNet fetch). After convert, the WFDB copy is deleted so only `.npy` / `.pkl` / `.json` remain.

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd() if (Path.cwd() / "evaluation" / "common.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "evaluation"))

import pandas as pd
from IPython.display import display

import common as C

# --- subset ---
N_RECORDS = 20          # set None for all 200 (dataset is only ~24 MB)
RECORD_IDS = None       # e.g. ["1", "33", "50"] to pin specific records
REQUIRE_ISCHEMIA = False
OVERWRITE_PROCESSED = False
CONSUME_WFDB = True
SEED = 42

RAW_DIR, PROC_DIR = C.dataset_dirs("ludb")
print("raw:", RAW_DIR)
print("processed:", PROC_DIR)


raw: C:\Users\staso\OneDrive\Pulpit\Nauka\University\Thesis\code\ECG_Delineation\WTdelineator\data\evaluation\raw\ludb
processed: C:\Users\staso\OneDrive\Pulpit\Nauka\University\Thesis\code\ECG_Delineation\WTdelineator\data\evaluation\processed\ludb


## 1. Catalogue + subset

In [6]:
catalogue = C.load_ludb_catalogue()
print(f"Catalogue rows: {len(catalogue)}")
display(catalogue.head())

pool = catalogue.copy()
if REQUIRE_ISCHEMIA:
    pool = pool[pool["Ischemia"].fillna("").astype(str).str.strip().ne("")]

if RECORD_IDS:
    wanted = {str(x).lstrip("0") for x in RECORD_IDS}
    selected = pool[pool["ID"].isin(wanted)]
else:
    n = len(pool) if N_RECORDS is None else min(int(N_RECORDS), len(pool))
    selected = pool.sample(n=n, random_state=SEED).sort_values("ID")

record_ids = [str(i) for i in selected["ID"].tolist()]
todo = C.unprocessed_ids(PROC_DIR, record_ids, overwrite=OVERWRITE_PROCESSED)
print(f"Selected {len(record_ids)}; already processed {len(record_ids) - len(todo)}; to acquire {len(todo)}")
print(record_ids)


Catalogue rows: 200


,ID,Sex,Age,Rhythms,Electric axis of the heart,Conduction abnormalities,Extrasystolies,Hypertrophies,Cardiac pacing,Ischemia,Non-specific repolarization abnormalities,Other states
0,1,F\n,51\n,Sinus bradycardia,Electric axis of the heart: left axis deviation,NaN,NaN,Left ventricular overload\nLeft ventricular hy...,NaN,NaN,Non-specific repolarization abnormalities: pos...,NaN
1,2,M\n,64\n,Sinus rhythm,Electric axis of the heart: normal,NaN,NaN,Left atrial hypertrophy\nLeft ventricular hype...,NaN,NaN,Non-specific repolarization abnormalities: pos...,NaN
2,3,M\n,53\n,Sinus rhythm,Electric axis of the heart: vertical,NaN,NaN,Left atrial hypertrophy\nLeft ventricular hype...,NaN,Ischemia: inferior wall\nIschemia: lateral wall,NaN,NaN
3,4,M\n,56\n,Sinus rhythm,Electric axis of the heart: left axis deviation,Incomplete right bundle branch block,NaN,Left atrial hypertrophy\nLeft ventricular hype...,NaN,Ischemia: inferior wall\nScar formation: infer...,NaN,NaN
4,5,M\n,61\n,Sinus rhythm,Electric axis of the heart: horizontal,NaN,NaN,Left atrial hypertrophy,NaN,NaN,Non-specific repolarization abnormalities: inf...,NaN


Selected 20; already processed 20; to acquire 0
['116', '129', '153', '159', '16', '166', '171', '175', '178', '183', '187', '31', '46', '57', '67', '69', '70', '79', '83', '96']


## 2. Convert (skip processed; consume WFDB)

In [ ]:
results = [
    C.acquire_and_convert_ludb(rid, overwrite=OVERWRITE_PROCESSED, consume=CONSUME_WFDB)
    for rid in record_ids
]
print(C.summarize_acquire(results))
failed = [r for r in results if r["status"] == "failed"]
if failed:
    print("failed:", failed)

index = C.load_index(PROC_DIR)
print(f"index rows: {len(index)}")
display(index.head())


## 3. Visualize

In [5]:
index = C.load_index(PROC_DIR)
example_id = str(index.iloc[0]["record_id"])
_ = C.plot_ludb_record(example_id, PROC_DIR)


In [3]:
index = C.load_index(PROC_DIR)
fields = [
    ("rhythm", "Rhythm"),
    ("axis", "Axis"),
    ("conduction", "Conduction"),
    ("extrasystoles", "Extrasystoles"),
    ("hypertrophy", "Hypertrophy"),
    ("pacing", "Pacing"),
    ("ischemia", "Ischemia"),
    ("nonspecific_repol", "Nonspecific repol."),
    ("other", "Other"),
]
present = pd.Series(
    {title: int(index[col].fillna("").astype(str).str.strip().ne("").sum()) for col, title in fields}
)
C.plot_category_counts(present, title="LUDB subset — records with each diagnostic family")
binary = pd.DataFrame({
    title: index[col].fillna("").astype(str).str.strip().ne("").astype(int)
    for col, title in fields
})
C.plot_cooccurrence(binary, title="LUDB subset — co-occurrence of diagnostic families")
